In [4]:
import pandas as pd
import glob
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

def run_etl():
    print("--- Starting ETL Pipeline ---")

    print("[1/4] Extracting data from CSV and JSON sources...")
    
    csv_files = glob.glob("raw_data/*.csv")
    df_csv = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
    
    df_json = pd.read_json("raw_data/sales_mar.json")
    
    df_staging = pd.concat([df_csv, df_json], ignore_index=True)
    print(f"Extracted {len(df_staging)} total records.")

    print("[2/4] Validating and removing invalid records...")
    
    df_valid = df_staging.dropna(subset=['transaction_id'])
    
    df_valid = df_valid[df_valid['price'] > 0]
    
    dropped_count = len(df_staging) - len(df_valid)
    print(f"Removed {dropped_count} invalid records.")

    print("[3/4] Transforming fields...")
    
    df_valid['category'] = df_valid['category'].str.upper()
    
    df_valid['transaction_date'] = pd.to_datetime(df_valid['transaction_date'])
    
    df_valid['total_value'] = df_valid['price'] * df_valid['quantity']
    
    df_valid['transaction_id'] = df_valid['transaction_id'].astype(int)

    print("[4/4] Executing incremental load to MS SQL Server...")
    
    server_name = r'DESKTOP-P5RT5RO'  
    database_name = 'DataEngineeringLab'
    table_name = 'Fact_Sales'
    
    connection_url = URL.create(
        "mssql+pyodbc",
        host=server_name,
        database=database_name,
        query={
            "driver": "ODBC Driver 17 for SQL Server",
            "Trusted_Connection": "yes"
        }
    )
    
    engine = create_engine(connection_url, fast_executemany=True)
    
    try:
        query = f'SELECT transaction_id FROM {table_name}'
        existing_ids = pd.read_sql(query, engine)['transaction_id'].tolist()
        
        df_new = df_valid[~df_valid['transaction_id'].isin(existing_ids)]
    except Exception:
        df_new = df_valid

    if not df_new.empty:
        df_new.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"SUCCESS: {len(df_new)} new records loaded into '{table_name}'.")
    else:
        print("SUCCESS: 0 new records to load. Database is up to date.")

    print("--- ETL Pipeline Complete ---")

if __name__ == "__main__":
    run_etl()

--- Starting ETL Pipeline ---
[1/4] Extracting data from CSV and JSON sources...
Extracted 8 total records.
[2/4] Validating and removing invalid records...
Removed 2 invalid records.
[3/4] Transforming fields...
[4/4] Executing incremental load to MS SQL Server...


c:\Users\admin\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1135.8'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


SUCCESS: 6 new records loaded into 'Fact_Sales'.
--- ETL Pipeline Complete ---
